In [1]:
from pathlib import Path
import json
from PIL import Image

# read the label file
IN_LABEL_DIR = Path(r'D:\UCSD_courses\ECE285\project\data\cleaned\valid\labels\cleaned_index.json')
IN_IMG_DIR = Path(r'D:\UCSD_courses\ECE285\project\data\cleaned\valid\images')
OUT_LABEL_DIR = Path(r'D:\UCSD_courses\ECE285\project\data\cleaned_yolo\labels\val')

with IN_LABEL_DIR.open("r", encoding="utf-8") as f:
    items = json.load(f)

missing_images = []
written = 0

for item in items:
    img_name = item["file_name"]
    img_path = IN_IMG_DIR / img_name

    if not img_path.exists():
        missing_images.append(img_name)
        continue

    # read actual image size
    with Image.open(img_path) as im:
        img_w, img_h = im.size

    x, y, w, h = item["bbox"]   # COCO format: top-left x,y,width,height

    # convert to YOLO normalized format
    xc = (x + w / 2) / img_w
    yc = (y + h / 2) / img_h
    w  = w / img_w
    h  = h / img_h

    # optional clamp
    xc = max(0.0, min(1.0, xc))
    yc = max(0.0, min(1.0, yc))
    w  = max(0.0, min(1.0, w))
    h  = max(0.0, min(1.0, h))

    label_path = OUT_LABEL_DIR / (Path(img_name).stem + ".txt")

    # use append in case multiple objects belong to same image
    with open(label_path, "a", encoding="utf-8") as f:
        f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        written += 1

print(f"Done. Wrote {written} label lines.")
print(f"Missing images: {len(missing_images)}")
if missing_images[:10]:
    print("First few missing:", missing_images[:10])

Done. Wrote 408 label lines.
Missing images: 0
